# Week 2 — Data Cleaning & Preprocessing

Task
Using the same dataset:

* Handle missing values
* Remove duplicates
* Encode categorical variables
* Normalize numerical features

### Step 1: Handle Missing Values:

The dataset contains missing values in the USD, EUR, GBP, INR, AED, and CNY columns. Missing values were replaced with the **median** value, calculated separately for each `Purity` group (24K/22K/18K), because gold price naturally differs by purity — filling with a single global median would distort that relationship.

### Step 2: Remove Duplicates:
Duplicate rows were identified using the `duplicated()` function and removed using `drop_duplicates()`. This ensures that no repeated records affect the analysis.
* Code Used: `df.drop_duplicates(inplace=True)`

### Step 3: Encode Categorical Variables
Unlike the earlier dataset, this dataset contains two real categorical columns: `Market` (city where the rate was recorded) and `Purity` (24K/22K/18K).
* `Purity` has a natural order, so it was **label encoded** (18K=0, 22K=1, 24K=2).
* `Market` has no natural order, so it was **one-hot encoded** using `pd.get_dummies()`.
* Code Used: `df['Purity_encoded'] = df['Purity'].map({'18K':0,'22K':1,'24K':2})`, `pd.get_dummies(df, columns=['Market'])`

### Step 4: Normalize Numerical Features
The dataset contains values with very different scales — USD values are in the hundreds while INR values are in the tens of thousands. To bring all features to a common scale, **Min-Max Normalization** was applied, transforming values to the range 0 to 1.

In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load dataset
df = pd.read_csv("annual_gold_rates.csv")

print("="*50)
print("ORIGINAL DATASET")
print("="*50)
print(df.head())
print("\nShape:", df.shape)

ORIGINAL DATASET
      Date    Market Purity     USD     EUR     GBP       INR      AED  \
0  1979-04   Kolkata    18K  231.76  140.77  108.84   1887.88   885.36   
1  2002-09   Kolkata    22K  280.67  297.25  186.83  13633.03  1030.90   
2  2000-08     Delhi    24K  281.11  304.92  185.67       NaN  1032.50   
3  1990-11   Chennai    24K  383.12  282.62  215.71   6689.93  1406.18   
4  2002-06  New York    24K  303.26  321.17  201.86  14729.96  1113.85   

       CNY  
0   704.86  
1  2323.10  
2  2327.15  
3  1825.02  
4  2510.02  

Shape: (1000, 9)


### Handling Missing Values

In [2]:
# ----------------------------------
# TASK 1: HANDLE MISSING VALUES
# ----------------------------------

print("\nTASK 1: HANDLE MISSING VALUES")

print("\nMissing Values Before:")
print(df.isnull().sum())

num_cols = ['USD', 'EUR', 'GBP', 'INR', 'AED', 'CNY']

# Fill using the median within each Purity group, since price
# genuinely differs by purity level
for col in num_cols:
    df[col] = df.groupby('Purity')[col].transform(lambda x: x.fillna(x.median()))

# Safety net in case a whole group had no valid values
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

print("\nMissing Values After:")
print(df.isnull().sum())


TASK 1: HANDLE MISSING VALUES

Missing Values Before:
Date       0
Market     0
Purity     0
USD       31
EUR       23
GBP       26
INR       28
AED       31
CNY       28
dtype: int64

Missing Values After:
Date      0
Market    0
Purity    0
USD       0
EUR       0
GBP       0
INR       0
AED       0
CNY       0
dtype: int64


### Removing Duplicate Values

In [3]:
# ----------------------------------
# TASK 2: REMOVE DUPLICATES
# ----------------------------------

print("\nTASK 2: REMOVE DUPLICATES")

print("Duplicate Rows Before:", df.duplicated().sum())

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

print("Duplicate Rows After:", df.duplicated().sum())
print("Shape after removing duplicates:", df.shape)


TASK 2: REMOVE DUPLICATES
Duplicate Rows Before: 36
Duplicate Rows After: 0
Shape after removing duplicates: (964, 9)


### Encode Categorical Variables

In [4]:
# ----------------------------------
# TASK 3: ENCODE CATEGORICAL VARIABLES
# ----------------------------------

print("\nTASK 3: ENCODE CATEGORICAL VARIABLES")

# Purity has a natural order (18K < 22K < 24K) -> label encoding
purity_order = {'18K': 0, '22K': 1, '24K': 2}
df['Purity_encoded'] = df['Purity'].map(purity_order)

# Market has no natural order -> one-hot encoding
df = pd.get_dummies(df, columns=['Market'], prefix='Market')

df.drop(columns=['Purity'], inplace=True)

print(df.head())


TASK 3: ENCODE CATEGORICAL VARIABLES
      Date     USD     EUR     GBP        INR      AED      CNY  \
0  1979-04  231.76  140.77  108.84   1887.880   885.36   704.86   
1  2002-09  280.67  297.25  186.83  13633.030  1030.90  2323.10   
2  2000-08  281.11  304.92  185.67  13010.775  1032.50  2327.15   
3  1990-11  383.12  282.62  215.71   6689.930  1406.18  1825.02   
4  2002-06  303.26  321.17  201.86  14729.960  1113.85  2510.02   

   Purity_encoded  Market_Chennai  Market_Delhi  Market_Dubai  Market_Kolkata  \
0               0           False         False         False            True   
1               1           False         False         False            True   
2               2           False          True         False           False   
3               2            True         False         False           False   
4               2           False         False         False           False   

   Market_London  Market_Mumbai  Market_New York  
0          False     

### Normalize Numerical features

In [5]:
# ----------------------------------
# TASK 4: NORMALIZE NUMERICAL FEATURES
# ----------------------------------

print("\nTASK 4: NORMALIZE NUMERICAL FEATURES")

scaler = MinMaxScaler()

df[num_cols] = scaler.fit_transform(df[num_cols])

print(df.head())


TASK 4: NORMALIZE NUMERICAL FEATURES
      Date       USD       EUR       GBP       INR       AED       CNY  \
0  1979-04  0.054227  0.034161  0.027133  0.000559  0.024829  0.003363   
1  2002-09  0.083578  0.141954  0.087130  0.089653  0.049491  0.144196   
2  2000-08  0.083842  0.147237  0.086237  0.084933  0.049762  0.144549   
3  1990-11  0.145060  0.131876  0.109347  0.036986  0.113085  0.100849   
4  2002-06  0.097135  0.158431  0.098692  0.097974  0.063548  0.160464   

   Purity_encoded  Market_Chennai  Market_Delhi  Market_Dubai  Market_Kolkata  \
0               0           False         False         False            True   
1               1           False         False         False            True   
2               2           False          True         False           False   
3               2            True         False         False           False   
4               2           False         False         False           False   

   Market_London  Market_Mumba

### Saving the cleaned Dataset

In [6]:
# ----------------------------------
# SAVE CLEAN DATASET
# ----------------------------------

df.to_csv("annual_gold_rate_cleaned.csv", index=False)

print("\nClean dataset saved as annual_gold_rate_cleaned.csv")


Clean dataset saved as annual_gold_rate_cleaned.csv
